# CartPole LQR and ROA Diagnostics

This notebook is the 4D CartPole counterpart of the LQR/ROA notebook. It linearizes the CartPole dynamics at the origin, computes an LQR Lyapunov quadratic, and evaluates it on 2D slices of the 4D state space.

In [ ]:
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import matplotlib.pyplot as plt
import numpy as np
from scipy.linalg import solve_continuous_are

EXAMPLE_DIR = os.getcwd()
if os.path.basename(EXAMPLE_DIR) != "4DCartPoler":
    EXAMPLE_DIR = os.path.join(EXAMPLE_DIR, "examples", "4DCartPoler")
PROJECT_ROOT = os.path.abspath(os.path.join(EXAMPLE_DIR, "../.."))

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
if EXAMPLE_DIR not in sys.path:
    sys.path.insert(0, EXAMPLE_DIR)

from SystemDynamics import f, G, Q, R


In [ ]:
def finite_difference_jacobian(func, x0, step=1e-6):
    x0 = np.asarray(x0, dtype=float)
    y0 = np.asarray(func(x0), dtype=float).reshape(-1)
    jac = np.zeros((len(y0), len(x0)))
    for i in range(len(x0)):
        xp = x0.copy()
        xm = x0.copy()
        xp[i] += step
        xm[i] -= step
        jac[:, i] = (np.asarray(func(xp)) - np.asarray(func(xm))) / (2 * step)
    return jac


x_eq = np.zeros(4)
A = finite_difference_jacobian(lambda x: f(*x), x_eq)
B_full = np.asarray(G(*x_eq), dtype=float)
B = B_full[:, [1]]
Q_lqr = np.eye(4)
R_lqr = np.array([[1e-4]])

P = solve_continuous_are(A, B, Q_lqr, R_lqr)
K = np.linalg.solve(R_lqr, B.T @ P)

print("A =")
print(A)
print("B =")
print(B)
print("P =")
print(P)
print("K =")
print(K)


In [ ]:
def V_lqr(x1, x2, x3, x4):
    x = np.stack([x1, x2, x3, x4], axis=0)
    return np.einsum("i...,ij,j...->...", x, P, x)


grid = np.linspace(-4.0, 4.0, 121)
X1, X2 = np.meshgrid(grid, grid, indexing="ij")
Z_x1x2 = V_lqr(X1, X2, 0.0, 0.0)
Z_x3x4 = V_lqr(0.0, 0.0, X1, X2)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
cs0 = axes[0].contourf(X1, X2, Z_x1x2, levels=40)
axes[0].set_title("LQR V slice: x3=0, x4=0")
axes[0].set_xlabel("x1")
axes[0].set_ylabel("x2")
fig.colorbar(cs0, ax=axes[0])

cs1 = axes[1].contourf(X1, X2, Z_x3x4, levels=40)
axes[1].set_title("LQR V slice: x1=0, x2=0")
axes[1].set_xlabel("x3")
axes[1].set_ylabel("x4")
fig.colorbar(cs1, ax=axes[1])
plt.show()
